# Kappa source-feature predictor analysis

## tl;dr

- The raw quotation-group difference is −12.94 percentage points (Welch p = 1.79e-06), but quotation does not improve prediction once broader source features are available.
- Greek vocabulary has the highest nested-CV R² (0.414); vocabulary plus log length has the lowest MAE (7.40%).
- Adding quotation to the all-available model changes MAE by +0.12 percentage points (95% paired-bootstrap CI −0.06 to +0.29), so there is no reliable incremental gain.


## Context & Methods

This notebook reproduces the source-only feature-block comparison published on the Kappa length-versus-quality page. The unit is one of the frozen 100 Kappa review entries. The outcome is the unweighted mean of BLEU-4, chrF++, METEOR, and ROUGE-L for one `gpt-5.6-sol` prompt-v3 translation against the approved house translation.

### Key Assumptions

- All predictors are available from the Greek source before translation; AI-output length and similarity scores are excluded from the predictors.
- Outer ten-fold cross-validation estimates held-out performance. Five-fold cross-validation inside each outer training fold selects the ridge penalty.
- Vocabulary, recogniser document frequency, imputation, and scaling are fitted within the training folds.
- Three source-version mismatches in the published rarity/segmentation snapshots are missing rather than reused; parsed dependency grammar has no coverage.


## Data

### 1. Load the published feature snapshot


In [1]:
from pathlib import Path
import sys
import pandas as pd

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from kappa_quality_predictor import (
    analyze_predictors,
    coverage_from_feature_rows,
    load_feature_snapshot,
)

feature_path = repo_root / "reference_site/statistics/kappa_quality_predictor_feature_snapshot.csv"
quotation_path = repo_root / "reference_site/statistics/kappa_quotation_quality_analysis.csv"
feature_rows = load_feature_snapshot(feature_path)
coverage = coverage_from_feature_rows(feature_rows)
coverage


{'row_count': 100,
 'feature_source': 'reviewed published source snapshots with live-audited source-version overrides',
 'rarity_run_id': None,
 'rarity_run_key': 'published feature snapshot',
 'rarity_current_count': 97,
 'sentence_segmentation_current_count': 97,
 'parsed_grammar_count': 0,
 'citation_entry_count': 95,
 'entity_entry_count': 100,
 'recogniser_entry_count': 97,
 'recogniser_match_rows': 819,
 'recogniser_detector_version': 'translation_guidance_scan_v4'}

### 2. Validate cohort and feature coverage


In [2]:
assert len(feature_rows) == 100
assert len({int(row["lemma_id"]) for row in feature_rows}) == 100
assert sorted(int(row["official_order"]) for row in feature_rows) == list(range(1, 101))
assert coverage["rarity_current_count"] == 97
assert coverage["sentence_segmentation_current_count"] == 97
pd.DataFrame([coverage]).T.rename(columns={0: "value"})


,value
row_count,100
feature_source,reviewed published source snapshots with live-...
rarity_run_id,None
rarity_run_key,published feature snapshot
rarity_current_count,97
sentence_segmentation_current_count,97
parsed_grammar_count,0
citation_entry_count,95
entity_entry_count,100
recogniser_entry_count,97


## Results

### 3. Refit the nested feature-block models


In [3]:
analysis = analyze_predictors(feature_rows)
model_table = pd.DataFrame(
    [{key: value for key, value in row.items() if key != "predictions"} for row in analysis["results"]]
)
model_table[[
    "family_label",
    "feature_count_median",
    "cv_r2",
    "cv_mae",
    "cv_rmse",
    "mae_improvement_vs_mean",
    "spearman_r",
]].sort_values("cv_r2", ascending=False).reset_index(drop=True)


,family_label,feature_count_median,cv_r2,cv_mae,cv_rmse,mae_improvement_vs_mean,spearman_r
0,Greek vocabulary,313,0.413568,0.075286,0.093918,0.025585,0.685433
1,Greek vocabulary + log length,314,0.402760,0.073983,0.094780,0.026887,0.665421
2,"All available source features, without quotation",398,0.345036,0.076374,0.099255,0.024496,0.616846
3,All available source features,402,0.340018,0.077534,0.099634,0.023336,0.626152
4,Source structure,7,0.301621,0.080464,0.102491,0.020406,0.552297
5,Log source length,1,0.274381,0.081711,0.104471,0.019159,0.534986
6,Log source length + quotation,4,0.255772,0.084257,0.105802,0.016613,0.525205
7,Guidance recogniser hits,66,0.137687,0.088963,0.113887,0.011907,0.457751
8,Quotation only,3,0.131224,0.094077,0.114313,0.006793,0.126822
9,Lexical rarity,3,0.067618,0.093052,0.118424,0.007818,0.357270


### 4. Inspect quotation's incremental value


In [4]:
quotation_table = pd.read_csv(quotation_path)
raw_result = quotation_table.loc[quotation_table["metric_key"] == "mean_lexical"].iloc[0]
summary = {
    "raw_difference_pp": raw_result["raw_difference"] * 100,
    "raw_welch_p": raw_result["raw_p_value"],
    "best_r2_family": analysis["best"]["family_label"],
    "best_r2": analysis["best"]["cv_r2"],
    "lowest_mae_family": analysis["best_mae"]["family_label"],
    "lowest_mae": analysis["best_mae"]["cv_mae"],
    **{f"all_quote_{key}": value for key, value in analysis["comparisons"]["all_quote"].items()},
}
pd.Series(summary, name="value").to_frame()


,value
raw_difference_pp,-12.940709
raw_welch_p,0.000002
best_r2_family,Greek vocabulary
best_r2,0.413568
lowest_mae_family,Greek vocabulary + log length
lowest_mae,0.073983
all_quote_delta_mae,0.00116
all_quote_delta_mae_ci_low,-0.00062
all_quote_delta_mae_ci_high,0.002918
all_quote_delta_r2,-0.005018


## Takeaways

The large raw quotation gap is real as a descriptive comparison, but quotation is strongly entangled with entry difficulty and length. Vocabulary carries the strongest generalisable signal; adding all available scalar and recogniser blocks to vocabulary reduces rather than improves held-out R². Quotation slightly worsens both the length model and the all-available model, with paired-bootstrap intervals crossing zero. The defensible conclusion is that quotation marks identify a difficult subset of this cohort but do not supply stable independent predictive information.

The next material extension is parsed source grammar. It should be added only after current-version analyses cover the cohort; the present 0/100 coverage does not support a grammar block.
